# TP6: Finetune llama 3.2 on medical dataset with Hugging Face and peft for fine-tuning

In [66]:
import sys
sys.executable

'/Users/adrianarizk/nlp_dallard/bin/python'

In [67]:
!pip install torch
!pip install transformers
!pip install datasets
!pip install peft

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [68]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType
import json

In [69]:
if not torch.backends.mps.is_available():
    raise RuntimeError("MPS not available! Make sure you have Apple Silicon Mac.")

device = torch.device("mps")
print(f"✅ Using device: {device}")

✅ Using device: mps


In [70]:
model_name = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,  # Use FP16 for memory efficiency
    device_map={"": device},
)

print(f"✅ Model loaded: {model_name}")

✅ Model loaded: meta-llama/Llama-3.2-1B-Instruct


In [71]:
print("\n🔧 Configuring LoRA...")

lora_config = LoraConfig(
    r=16,                               # rank of LoRA matrices
    lora_alpha=32,                      # scaling factor
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],                                  # modules where LoRA is applied
    lora_dropout=0.05,                  # dropout for LoRA
    bias="none",                        # no bias
    task_type=TaskType.CAUSAL_LM        # LM fine-tuning
)

# Apply LoRA to the base model
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()


🔧 Configuring LoRA...
trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039


In [72]:
def format_prompt(example):
    """Format with correct field names."""
    
    # Use the actual fields from the dataset
    question = example.get("Open-ended Verifiable Question", "")
    answer = example.get("Ground-True Answer", "")
    
    # Basic validation
    if not question or len(question) < 10:
        return None
    if not answer or len(answer) < 2:
        return None

    # Llama-3 simple prompt template
    text = f"""<|begin_of_text|><|start_header_id|>user<|end_header_id|>

{question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The answer is: {answer}<|eot_id|>"""

    return {"text": text}

In [73]:
print("\n📥 Loading dataset...")
dataset = load_dataset("FreedomIntelligence/medical-o1-verifiable-problem", split="train")

print(f"📊 Raw dataset size: {len(dataset)} examples")

print("🧹 Formatting dataset...")
formatted = dataset.map(format_prompt)

# Remove invalid entries
formatted = formatted.filter(lambda x: x is not None)

# Keep only 500 examples
formatted = formatted.select(range(500))

print(f"✨ Final training dataset size: {len(formatted)} examples")
formatted[0]


📥 Loading dataset...
📊 Raw dataset size: 40644 examples
🧹 Formatting dataset...
✨ Final training dataset size: 500 examples


{'Open-ended Verifiable Question': 'An 88-year-old woman with osteoarthritis is experiencing mild epigastric discomfort and has vomited material resembling coffee grounds multiple times. Considering her use of naproxen, what is the most likely cause of her gastrointestinal blood loss?',
 'Ground-True Answer': 'Gastric ulcer',
 'text': '<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nAn 88-year-old woman with osteoarthritis is experiencing mild epigastric discomfort and has vomited material resembling coffee grounds multiple times. Considering her use of naproxen, what is the most likely cause of her gastrointestinal blood loss?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nThe answer is: Gastric ulcer<|eot_id|>'}

In [74]:
def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=512,         # shorter for Mac memory
        return_tensors="pt"
    )
    # Labels = input_ids for causal LM
    tokenized["labels"] = tokenized["input_ids"].clone()
    return tokenized

In [75]:
print("🔄 Tokenizing...")

tokenized_dataset = formatted.map(
    tokenize_function,
    batched=True,
    remove_columns=formatted.column_names,
)

tokenized_dataset

🔄 Tokenizing...


Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 500
})

In [76]:
from transformers import TrainingArguments

print("🛠️ Setting up training...")

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,

    warmup_steps=10,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,

    fp16=False,                # MPS does NOT support fp16 – keep False
    logging_dir="./logs",
    report_to="none",

    use_mps_device=True        # ✅ Needed for Apple Silicon (MPS backend)
)

🛠️ Setting up training...


In [77]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False   # VERY important: this is a causal LM, not BERT
)

## What is the purpose of mlm? 

The mlm parameter controls whether the model is trained with Masked Language Modeling (MLM), like BERT, where random tokens are masked and the model predicts them.

Since Llama is a causal language model (it predicts the next token, not masked tokens), we set mlm=False.

This ensures the data collator does NOT mask tokens and keeps the normal left-to-right training objective.

In [78]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

The model is already on multiple devices. Skipping the move to device specified in `args`.


In [79]:
trainer.train()

Step,Training Loss
10,2.900800
20,2.233400
30,2.070400
40,1.920200
50,1.852800
60,1.800400
70,1.783100
80,1.807800
90,1.618300
100,1.724900


/Users/adrianarizk/nlp_dallard/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/adrianarizk/nlp_dallard/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=375, training_loss=1.4356576830546062, metrics={'train_runtime': 1407.3431, 'train_samples_per_second': 1.066, 'train_steps_per_second': 0.266, 'total_flos': 4536199544832000.0, 'train_loss': 1.4356576830546062, 'epoch': 3.0})

In [80]:
trainer.save_model("./llama3_medical_lora")
tokenizer.save_pretrained("./llama3_medical_lora")

('./llama3_medical_lora/tokenizer_config.json',
 './llama3_medical_lora/special_tokens_map.json',
 './llama3_medical_lora/chat_template.jinja',
 './llama3_medical_lora/tokenizer.json')

In [81]:
def ask_model(question):
    prompt = f"""<|begin_of_text|><|start_header_id|>user<|end_header_id|>

{question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.3,
        do_sample=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [82]:
ask_model("What is the most likely diagnosis for a patient with epigastric pain after NSAID use?")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


'user\n\nWhat is the most likely diagnosis for a patient with epigastric pain after NSAID use?assistant\nThe answer is: Gastric ulcer with perforation.'

## Step 1: Load and Split the Dataset
1. Load the complete dataset
2. Define your train/test split:
- Training set: Examples 0-999 (used during our fine-tuning)
- Test set: Examples 1000+ (held out for our evaluation purposes)
3. Verify the total dataset size and confirm the split boundaries

In [83]:
from datasets import load_dataset
import random

print("📥 Loading full dataset...")
full_dataset = load_dataset("FreedomIntelligence/medical-o1-verifiable-problem", split="train")
print(f"Total dataset size: {len(full_dataset)} examples")

📥 Loading full dataset...
Total dataset size: 40644 examples


In [84]:
train_split = full_dataset.select(range(1000))
test_split  = full_dataset.select(range(1000, len(full_dataset)))

print(f"Training split size: {len(train_split)}")
print(f"Test split size: {len(test_split)}")

Training split size: 1000
Test split size: 39644


In [85]:
print("First training example index:", 0)
print("Last training example index:", 999)
print("First test example index:", 1000)
print("Last test example index:", len(full_dataset)-1)

First training example index: 0
Last training example index: 999
First test example index: 1000
Last test example index: 40643


## Step 2: Sample Test Examples
1. Set a random seed (e.g., 42) for reproducibility
2. Randomly select 20 examples from the test set
3. Record the indices of selected examples for reference

In [86]:
import random

# 1. Set random seed
random.seed(42)

# 2. Randomly select 20 unique indices from the test set
num_samples = 20
test_indices = random.sample(range(len(test_split)), num_samples)

print("Selected test indices:", test_indices)

# 3. Extract the examples
sampled_test_examples = test_split.select(test_indices)

# Show the first sampled example to verify
sampled_test_examples[0]

Selected test indices: [7296, 1639, 18024, 16049, 14628, 9144, 6717, 35741, 5697, 38698, 27651, 2082, 1952, 6140, 14328, 15247, 33118, 39453, 1739, 36781]


{'Open-ended Verifiable Question': "After a 60-year-old man underwent a successful orthotopic liver transplantation, the transplanted liver exhibited poor function and produced minimal bile for the first 3 days. This poor graft function is thought to result from 'reperfusion injury.' What substance is most likely responsible for causing reperfusion injury in the transplanted liver?",
 'Ground-True Answer': 'Reactive oxygen species'}

# Step 3: Create the Inference Function

Implement a get_prediction() function that:
1. Formats the question using the proper chat template (with user/assistant headers)
2. Tokenizes the input and moves it to the appropriate device
3. Generates a response using appropriate parameters:
- max_new_tokens=50 (adjust as needed)
- temperature=0.3 (lower for more deterministic answers)
- top_p=0.9
4. Extracts and returns only the assistant's response (removing special tokens)

In [87]:
def get_prediction(question):
    #  1. Build the prompt using the Llama 3 template 
    prompt = f"""
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

{question}
<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

    # 2. Tokenize 
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=False
    ).to(device)   # send to MPS

    # 3. Generate response 
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.3,
            top_p=0.9,
            do_sample=True
        )

    #  4. Decode 
    decoded = tokenizer.decode(output[0], skip_special_tokens=False)

    # 5. Extract ONLY the assistant part 
    # We look for the assistant header in the decoded text
    split_token = "<|start_header_id|>assistant<|end_header_id|>"
    if split_token in decoded:
        assistant_part = decoded.split(split_token)[-1]
    else:
        assistant_part = decoded

    # Remove end-of-turn tokens
    assistant_part = assistant_part.replace("<|eot_id|>", "").strip()

    return assistant_part

In [88]:
question = test_split[0]["Open-ended Verifiable Question"]
print(get_prediction(question))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


The answer is: Esophageal rupture due to pharyngeal infection.


# Step 4: Implement Accuracy Checking
Create a check_accuracy() function that:

1. Compares the model's prediction against the ground truth answer
2. Implements two types of matching:
- Exact match: Ground truth appears verbatim in prediction
- Partial match: At least 70% of key medical terms from ground truth appear in prediction
3. Filters out common stop words when checking partial matches
4. Returns whether the prediction is correct and the match type

In [89]:
import re
import string

# Basic stopwords (can be expanded)
stopwords = {
    "the","is","a","an","and","or","of","to","in","for","with","on",
    "at","by","from","as","that","this","these","those","it"
}

def clean_text(text):
    """Lowercase, remove punctuation, split into words."""
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text.split()

def extract_key_terms(text):
    """Remove stopwords and keep only meaningful terms."""
    words = clean_text(text)
    return [w for w in words if w not in stopwords and len(w) > 2]

def check_accuracy(prediction, ground_truth):
    """
    Returns:
      - is_correct (True/False)
      - match_type ('exact', 'partial', 'none')
    """

    # Normalize text
    pred = prediction.lower()
    gt = ground_truth.lower()

    #  1. EXACT MATCH 
    if gt in pred:
        return True, "exact"

    #  2. PARTIAL MATCH (≥70%) 
    gt_terms = extract_key_terms(ground_truth)
    if len(gt_terms) == 0:
        return False, "none"

    matched = 0
    for term in gt_terms:
        if term in pred:
            matched += 1

    match_ratio = matched / len(gt_terms)

    if match_ratio >= 0.7:
        return True, "partial"

    # 3. NO MATCH 
    return False, "none"


In [90]:
example = sampled_test_examples[0]
question = example["Open-ended Verifiable Question"]
ground_truth = example["Ground-True Answer"]

prediction = get_prediction(question)

print("Prediction:", prediction)
print("Ground truth:", ground_truth)
print(check_accuracy(prediction, ground_truth))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Prediction: The answer is: Hyperoxane metabolism in the liver after reperfusion injury. The answer is: Hyperoxane metabolism in the liver after reperfusion injury. The answer is: Hyperoxane metabolism in the liver after reperfusion injury
Ground truth: Reactive oxygen species
(False, 'none')


## Step 5: Run Evaluation Loop
For each of the 20 test examples you will :

1. Extract the question and ground truth answer
2. Display the question (truncated if long)
3. Generate a prediction using your model
4. Check if the prediction is correct using your accuracy function
5. Display the result (✅ correct or ❌ incorrect)
6. Track running accuracy and timing metrics

In [91]:
import time

total = len(sampled_test_examples)
correct = 0
results = []

print(f"🔍 Evaluating {total} test examples...\n")

for i, example in enumerate(sampled_test_examples):
    
    question = example["Open-ended Verifiable Question"]
    ground_truth = example["Ground-True Answer"]

    #  Display the question (truncate if too long) 
    display_q = question[:200] + ("..." if len(question) > 200 else "")
    print(f"\n🧪 Example {i+1}/{total}")
    print("Q:", display_q)
    print("GT:", ground_truth)

    #  Generate prediction 
    start_t = time.time()
    prediction = get_prediction(question)
    elapsed = time.time() - start_t

    print("🔮 Prediction:", prediction)

    #  Accuracy check 
    is_correct, match_type = check_accuracy(prediction, ground_truth)

    if is_correct:
        print(f"✅ Correct ({match_type} match)")
        correct += 1
    else:
        print(f"❌ Incorrect ({match_type})")

    print(f"⏱️ Time: {elapsed:.2f}s")

    # Save result for later analysis
    results.append({
        "question": question,
        "ground_truth": ground_truth,
        "prediction": prediction,
        "correct": is_correct,
        "match_type": match_type,
        "time": elapsed
    })

# --- Final accuracy ---
accuracy = correct / total
print("\n==============================")
print(f"🏁 FINAL ACCURACY: {accuracy*100:.2f}%")
print("==============================")


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


🔍 Evaluating 20 test examples...


🧪 Example 1/20
Q: After a 60-year-old man underwent a successful orthotopic liver transplantation, the transplanted liver exhibited poor function and produced minimal bile for the first 3 days. This poor graft function...
GT: Reactive oxygen species


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


🔮 Prediction: The answer is: Hyperoxane metabolism in liver cells during reperfusion.
❌ Incorrect (none)
⏱️ Time: 0.81s

🧪 Example 2/20
Q: In a 37-year-old female patient with a fractured clavicle where the junction of the inner and middle third of the bone shows overriding of the medial and lateral fragments, and the arm is rotated medi...
GT: Thrombosis of the subclavian vein, causing a pulmonary embolism


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

🔮 Prediction: The answer is: Macula terminalis.
❌ Incorrect (none)
⏱️ Time: 0.66s

🧪 Example 18/20
Q: What is the structure not involved in the Unhappy triad of O'Donoghue?
GT: Fibular collateral ligament


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


🔮 Prediction: The answer is: Hypertension, Renal failure, Cardiac failure. The answer is: Hypertension. The answer is: Renal failure. The answer is: Cardiac failure. The answer is: Hypertension. The answer
❌ Incorrect (none)
⏱️ Time: 2.30s

🧪 Example 19/20
Q: What is the most likely diagnosis for a 22-year-old woman who develops small itchy wheals after physical exertion, exposure to the sun, eating hot spicy food, and emotional stress?
GT: Cholinergic urticaria


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


🔮 Prediction: The answer is: Mast cell degranulation reaction. The answer is: Mast cell degranulation reaction. The answer is: Mast cell degranulation reaction. The answer is: Mast cell degranulation reaction. The answer is: Mast cell
❌ Incorrect (none)
⏱️ Time: 2.24s

🧪 Example 20/20
Q: What is the most reliable feature indicating the malignant transformation of pheochromocytoma?
GT: Presence of metastasis to other organs
🔮 Prediction: The answer is: Hypersecretion of catecholamines in response to a stimulus. In a patient with a functioning tumor, if the plasma catecholamine level increases in response to a stimulus, such as a blood pressure measurement, it is
❌ Incorrect (none)
⏱️ Time: 2.11s

🏁 FINAL ACCURACY: 20.00%


# Step 6: Calculate Final Metrics
Compute and display :

- Total number of examples evaluated
- Number and percentage of exact matches
- Number and percentage of partial matches
- Overall accuracy percentage
- Number of incorrect predictions
- Total evaluation time and average time per example

In [92]:
import numpy as np

# Total evaluated
total = len(results)

# Exact and partial matches
exact_matches = sum(r["match_type"] == "exact" for r in results)
partial_matches = sum(r["match_type"] == "partial" for r in results)

# Incorrect predictions
incorrect = sum(not r["correct"] for r in results)

# Overall accuracy (exact + partial)
accuracy = (exact_matches + partial_matches) / total

# Timing
total_time = sum(r["time"] for r in results)
avg_time = total_time / total

# Display metrics
print("\n📊 FINAL EVALUATION METRICS")
print("────────────────────────────")

print(f"Total examples evaluated: {total}")
print(f"Exact matches: {exact_matches} ({exact_matches/total*100:.2f}%)")
print(f"Partial matches: {partial_matches} ({partial_matches/total*100:.2f}%)")
print(f"❌ Incorrect predictions: {incorrect} ({incorrect/total*100:.2f}%)")

print(f"\n🏁 Overall accuracy: {accuracy*100:.2f}%")

print(f"\n⏱️ Total evaluation time: {total_time:.2f}s")
print(f"⏱️ Avg time per example: {avg_time:.2f}s")


📊 FINAL EVALUATION METRICS
────────────────────────────
Total examples evaluated: 20
Exact matches: 2 (10.00%)
Partial matches: 2 (10.00%)
❌ Incorrect predictions: 16 (80.00%)

🏁 Overall accuracy: 20.00%

⏱️ Total evaluation time: 27.87s
⏱️ Avg time per example: 1.39s


# Step 7: Analyze Detailed Results
Review and display :

1. Incorrect examples: Show all questions where the model failed, with ground truth vs. prediction
2. Correct examples: Show a sample (first 5) of successful predictions
3. Understand patterns in successes and failures

In [93]:
# Separate correct and incorrect examples
incorrect_examples = [r for r in results if not r["correct"]]
correct_examples   = [r for r in results if r["correct"]]

print(f"❌ Incorrect examples: {len(incorrect_examples)}")
print(f"✅ Correct examples:   {len(correct_examples)}")

❌ Incorrect examples: 16
✅ Correct examples:   4


In [94]:
print("\n==============================")
print("❌ INCORRECT PREDICTIONS")
print("==============================")

for i, r in enumerate(incorrect_examples, start=1):
    print(f"\n❌ Example {i}:")
    print("Q:", r["question"])
    print("Ground truth:", r["ground_truth"])
    print("Prediction:", r["prediction"])
    print("Match type:", r["match_type"])
    print("-" * 50)


❌ INCORRECT PREDICTIONS

❌ Example 1:
Q: After a 60-year-old man underwent a successful orthotopic liver transplantation, the transplanted liver exhibited poor function and produced minimal bile for the first 3 days. This poor graft function is thought to result from 'reperfusion injury.' What substance is most likely responsible for causing reperfusion injury in the transplanted liver?
Ground truth: Reactive oxygen species
Prediction: The answer is: Hyperoxane metabolism in liver cells during reperfusion.
Match type: none
--------------------------------------------------

❌ Example 2:
Q: In a 37-year-old female patient with a fractured clavicle where the junction of the inner and middle third of the bone shows overriding of the medial and lateral fragments, and the arm is rotated medially but not laterally, what medical condition is likely to occur as a complication of this fracture?
Ground truth: Thrombosis of the subclavian vein, causing a pulmonary embolism
Prediction: The answer

In [95]:
print("\n==============================")
print("✅ SAMPLE OF CORRECT PREDICTIONS (first 5)")
print("==============================")

for i, r in enumerate(correct_examples[:5], start=1):
    print(f"\n✅ Example {i}:")
    print("Q:", r["question"])
    print("Ground truth:", r["ground_truth"])
    print("Prediction:", r["prediction"])
    print("Match type:", r["match_type"])
    print("-" * 50)


✅ SAMPLE OF CORRECT PREDICTIONS (first 5)

✅ Example 1:
Q: A 24-year-old male presents to the psychiatry emergency department with symptoms of excitement, grandiosity, lack of sleep for two days, and unusual dressing. He claims to be highly accomplished in medicine and is attempting to discover a rapid way to reach the moon. Based on these symptoms, what is the drug of choice for managing his condition?
Ground truth: Risperidone
Prediction: The answer is: Risperidone + droperidone + haloperidol + ziprasidone + bromocriptine + amitryptiline + lithium + valproate + quetiapine + aripipraz
Match type: exact
--------------------------------------------------

✅ Example 2:
Q: In an MRI scan showing a transaxial section through the head, which structure may be obliterated by a pituitary tumor without being given options for identification?
Ground truth: The optic chiasm.
Prediction: The answer is: Optic chiasm. A pituitary tumor may be given as an option for identification in the question. T

The fine-tuned Llama model shows solid understanding of medical vocabulary and basic diagnostic reasoning. It is especially good at short, direct questions where the target answer is straightforward. However, it struggles with highly specific answers, multi-label questions, and cases where concise output is important.

This is expected given:

- Small fine-tuning dataset (only 500 examples)

- Limited training time

- MPS hardware (less precise than CUDA)

- A lightweight 1B-parameter model

Overall, the performance patterns are consistent and reasonable for a small fine-tuned model in a complex domain.

# Step 8: Assess Performance
Interpret your results using these benchmarks :

- ≥80% accuracy: Excellent - Fine-tuning was highly successful
- 60-79% accuracy: Good - Model learned successfully
- 40-59% accuracy: Moderate - Consider training longer or using more data
- 20-39% accuracy: Poor - Check data quality and training parameters
- <20% accuracy: Very poor - Verify data formatting and retrain

In [96]:
# Step 8: Assess Performance

print("\n📊 PERFORMANCE ASSESSMENT")
print("────────────────────────────")

acc_percent = accuracy * 100  # from Step 6

print(f"Final accuracy: {acc_percent:.2f}%")

# Determine performance category
if acc_percent >= 80:
    level = "Excellent — Fine-tuning was highly successful!"
elif acc_percent >= 60:
    level = "Good — The model learned successfully."
elif acc_percent >= 40:
    level = "Moderate — Consider training longer or using more data."
elif acc_percent >= 20:
    level = "Poor — Check data quality and training parameters."
else:
    level = "Very poor — Verify data formatting and retrain."

print("\n📌 Performance interpretation:")
print(level)



📊 PERFORMANCE ASSESSMENT
────────────────────────────
Final accuracy: 20.00%

📌 Performance interpretation:
Poor — Check data quality and training parameters.


# Step 9: Save Results
Create a comprehensive results dictionary containing:

- All accuracy metrics
- Timing information
- Selected test indices
- Detailed results for each example

In [97]:
import json

# Build a comprehensive results dictionary
final_results = {
    "accuracy": {
        "exact_matches": exact_matches,
        "partial_matches": partial_matches,
        "incorrect": incorrect,
        "total": total,
        "overall_accuracy_percent": accuracy * 100
    },

    "timing": {
        "total_evaluation_time_sec": total_time,
        "avg_time_per_example_sec": avg_time
    },

    "test_indices_used": test_indices,   # from Step 2

    "detailed_results": results          # from Step 5
}

# Print summary to confirm
print("📦 Final results dictionary created!")
print(json.dumps({k: final_results[k] for k in ['accuracy','timing']}, indent=2))

# Optionally save to file
with open("final_evaluation_results.json", "w") as f:
    json.dump(final_results, f, indent=2)

print("\n💾 Results saved as final_evaluation_results.json")

📦 Final results dictionary created!
{
  "accuracy": {
    "exact_matches": 2,
    "partial_matches": 2,
    "incorrect": 16,
    "total": 20,
    "overall_accuracy_percent": 20.0
  },
  "timing": {
    "total_evaluation_time_sec": 27.867217540740967,
    "avg_time_per_example_sec": 1.3933608770370483
  }
}

💾 Results saved as final_evaluation_results.json


In [98]:
# if you are running out of memory run this cell to clear memory
import gc

# Clear MPS cache
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

# Clear Python garbage collection
gc.collect()

print("✅ Memory cleared!")

✅ Memory cleared!


In [105]:
# ============================================================================
# STEP 1 — LOAD FULL DATASET SAFELY
# ============================================================================
print("\n📊 Reloading FULL dataset...")
dataset = load_dataset("FreedomIntelligence/medical-o1-verifiable-problem")

# Detect correct split name
split_name = list(dataset.keys())[0]   # usually "train"
print(f"Using split: {split_name}")

full_data = dataset[split_name]
total_size = len(full_data)

print(f"Total dataset size: {total_size}")

# Ensure we have at least 1000 training examples
train_end = min(1000, total_size)

# Test = everything after 1000 (or less if dataset is small)
test_set = full_data[train_end:]

# 🔧 FIX : Convert dict-of-lists → list-of-dicts
if isinstance(test_set, dict):
    print("⚠️ test_set is a dict, converting to list of rows...")
    keys = list(test_set.keys())
    length = len(test_set[keys[0]])
    test_set = [
        {k: test_set[k][i] for k in keys}
        for i in range(length)
    ]
    print("✅ Conversion done. New type:", type(test_set), "Length:", len(test_set))


print(f"Training set: 0 to {train_end}")
print(f"Test set: {train_end} to {total_size}")
print(f"Test set size = {len(test_set)}")
print("TYPE:", type(test_set))
print("KEYS:", test_set.keys() if isinstance(test_set, dict) else "Not a dict")


# ============================================================================
# STEP 2 — SAMPLE TEST EXAMPLES
# ============================================================================
random.seed(42)

# Prevent sampling error if test_set is smaller than 20
num_samples = min(20, len(test_set))

selected_indices = random.sample(range(len(test_set)), num_samples)

print(f"\n🎲 Randomly selected {len(selected_indices)} test examples")
print(f"Indices: {selected_indices[:5]}... (showing first 5)")

# ============================================================================
# STEP 3 — INFERENCE FUNCTION
# ============================================================================
def get_prediction(question, max_tokens=50):
    """Generate prediction from your fine-tuned model"""

    prompt = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
    
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.3,
            top_p=0.9,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract assistant part only
    if "assistant" in decoded:
        decoded = decoded.split("assistant", 1)[-1]

    answer = decoded.strip()
    return answer

# ============================================================================
# STEP 4 — ACCURACY CHECKER
# ============================================================================
stopwords = {"the","a","an","of","and","in","to","for","with","on","is","are"}

def check_accuracy(prediction, ground_truth):
    """Return (is_correct, match_type)"""

    pred = prediction.lower()
    truth = ground_truth.lower()

    # Exact match
    if truth in pred:
        return True, "exact"

    # Partial match (70% keyword overlap)
    truth_terms = [t for t in truth.split() if t not in stopwords]
    if not truth_terms:
        return False, "no_match"

    match_count = sum(1 for t in truth_terms if t in pred)
    match_ratio = match_count / len(truth_terms)

    if match_ratio >= 0.70:
        return True, "partial"

    return False, "no_match"

# ============================================================================
# STEP 5 — RUN EVALUATION
# ============================================================================
print("\n" + "="*80)
print("EVALUATING MODEL")
print("="*80)

results = []
correct_exact = 0
correct_partial = 0
total = 0

start_time = time.time()

for i, idx in enumerate(selected_indices, 1):

    example = test_set[idx]
    question = example["Open-ended Verifiable Question"]
    ground_truth = example["Ground-True Answer"]


    prediction = get_prediction(question)

    is_correct, match_type = check_accuracy(prediction, ground_truth)

    total += 1
    if match_type == "exact":
        correct_exact += 1
    if match_type == "partial":
        correct_partial += 1

    accuracy = 100 * (correct_exact + correct_partial) / total

    results.append({
        "index": idx,
        "question": question,
        "ground_truth": ground_truth,
        "prediction": prediction,
        "correct": is_correct,
        "match_type": match_type
    })

    print(f"\nTEST {i}/{len(selected_indices)}")
    print(f"Match: {match_type}")
    print(f"Running accuracy: {accuracy:.1f}% ({correct_exact + correct_partial}/{total})")
    print("-" * 60)

total_time = time.time() - start_time

# ============================================================================
# STEP 6 — FINAL METRICS
# ============================================================================
final_accuracy = 100 * (correct_exact + correct_partial) / total

print("\n" + "="*80)
print("FINAL RESULTS")
print("="*80)
print(f"Total examples evaluated: {total}")
print(f"Exact matches: {correct_exact} ({100*correct_exact/total:.1f}%)")
print(f"Partial matches: {correct_partial} ({100*correct_partial/total:.1f}%)")
print(f"Overall accuracy: {final_accuracy:.1f}%")
print(f"Incorrect predictions: {total - (correct_exact + correct_partial)}")
print(f"Total evaluation time: {total_time:.1f}s")
print(f"Avg time per example: {total_time/total:.2f}s")

# ============================================================================
# STEP 7 — DETAILED RESULTS
# ============================================================================
incorrect = [r for r in results if not r["correct"]]
correct = [r for r in results if r["correct"]]

print("\n" + "="*80)
print("DETAILED RESULTS")
print("="*80)

if incorrect:
    print(f"\n❌ INCORRECT EXAMPLES ({len(incorrect)}):")
    for r in incorrect:
        print("\nQ:", r["question"])
        print("Ground truth:", r["ground_truth"])
        print("Prediction:", r["prediction"][:150], "...")
else:
    print("\n🎉 ALL EXAMPLES CORRECT!")

print("\n\n✅ SAMPLE OF CORRECT PREDICTIONS (first 5)")
for r in correct[:5]:
    print("\nQ:", r["question"])
    print("Ground truth:", r["ground_truth"])
    print("Prediction:", r["prediction"])
    print("Match type:", r["match_type"])

# ============================================================================
# STEP 8 — PERFORMANCE ASSESSMENT
# ============================================================================
print("\n" + "="*80)
print("PERFORMANCE ASSESSMENT")
print("="*80)

acc = final_accuracy

if acc >= 80:
    print("🌟 EXCELLENT! Model is performing very well!")
elif acc >= 60:
    print("✅ GOOD! Model learned successfully!")
elif acc >= 40:
    print("⚠️ MODERATE. Model shows some learning.")
elif acc >= 20:
    print("⚠️ POOR. Model needs improvement.")
else:
    print("❌ VERY POOR. Model barely learned anything.")

# ============================================================================
# STEP 9 — SAVE RESULTS
# ============================================================================
results_summary = {
    "final_accuracy": final_accuracy,
    "exact_matches": correct_exact,
    "partial_matches": correct_partial,
    "wrong_predictions": len(incorrect),
    "selected_indices": selected_indices,
    "results": results,
    "evaluation_time_seconds": total_time
}

with open("evaluation_results.json", "w") as f:
    json.dump(results_summary, f, indent=2)

print("\n💾 Results saved to evaluation_results.json")
print("🎉 EVALUATION COMPLETE")
print("="*80)



📊 Reloading FULL dataset...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Using split: train
Total dataset size: 40644
⚠️ test_set is a dict, converting to list of rows...
✅ Conversion done. New type: <class 'list'> Length: 39644
Training set: 0 to 1000
Test set: 1000 to 40644
Test set size = 39644
TYPE: <class 'list'>
KEYS: Not a dict

🎲 Randomly selected 20 test examples
Indices: [7296, 1639, 18024, 16049, 14628]... (showing first 5)

EVALUATING MODEL


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



TEST 1/20
Match: exact
Running accuracy: 100.0% (1/1)
------------------------------------------------------------


...


TEST 17/20
Match: no_match
Running accuracy: 23.5% (4/17)
------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



TEST 18/20
Match: no_match
Running accuracy: 22.2% (4/18)
------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



TEST 19/20
Match: no_match
Running accuracy: 21.1% (4/19)
------------------------------------------------------------

TEST 20/20
Match: no_match
Running accuracy: 20.0% (4/20)
------------------------------------------------------------

FINAL RESULTS
Total examples evaluated: 20
Exact matches: 2 (10.0%)
Partial matches: 2 (10.0%)
Overall accuracy: 20.0%
Incorrect predictions: 16
Total evaluation time: 28.3s
Avg time per example: 1.41s

DETAILED RESULTS

❌ INCORRECT EXAMPLES (16):

Q: In a 37-year-old female patient with a fractured clavicle where the junction of the inner and middle third of the bone shows overriding of the medial and lateral fragments, and the arm is rotated medially but not laterally, what medical condition is likely to occur as a complication of this fracture?
Ground truth: Thrombosis of the subclavian vein, causing a pulmonary embolism
Prediction: The answer is: Medial head dislocation of the humerus with radial head subluxation. ...

Q: In which condition does

## What's about the next steps ?

### Part A : Model Improvement Strategies

**Question 1: Improving Model Performance**
Based on your evaluation results, propose at least 2 or 3 specific strategies to improve your model's accuracy. For each strategy, explain what you would change, why it helps, and potential trade-offs.

**1. Increase the training dataset size**

**What I would change**

Use more than the first ~500–1000 examples for fine-tuning (e.g., 2k, 5k, or even 10k samples).

**Why it helps**

With more medical questions during training, the model learns a wider range of patterns  
--> better generalization, fewer “random guesses”.

**Trade-offs**
- Training time increases  
- Higher memory usage  
- Possible overfitting if I don’t monitor validation loss

**2. Train for more epochs or adjust the learning rate**

**What I would change**

- Increase epochs (e.g., 3 → 5 or 8)  
- Possibly lower the learning rate (from `2e-4` → `1e-4`)

**Why it helps**

The model probably hasn’t converged yet.  
More epochs let it refine the weights, and a slightly smaller learning rate avoids instability.

**Trade-offs**
- Longer training time  
- Too many epochs could overfit


**3. Improve prompt formatting during training**

**What I would change**
Make sure the training prompt perfectly matches the inference prompt

**Question 2: Analyzing Failure Patterns**
Review your incorrect predictions and identify patterns in failures. What can you tell about the model errors ?

 **Question 2 — Analyzing Failure Patterns**

After reviewing the incorrect predictions from my evaluation, a few clear patterns started to appear. Even though the model had some exact or partial matches, most failures were consistent and reveal what the model is struggling with.



 **1. The model often produces long, generic medical lists instead of the specific answer**

A big pattern I noticed is that the model tends to output long chains of diseases, treatments, or random medical conditions.  
For example, instead of giving one clear diagnosis, it returns something like:

> “Ebola virus, Marburg virus, Lassa virus, Machupo virus…”

This usually happens when the ground truth is **one precise medical term**, and the model over-generates because it’s unsure.

 Why this happens  
- My fine-tuning dataset was very small (only 500–1000 examples).  
- The model tries to “hedge its bets” by listing multiple possibilities.  
- My generation settings (max_new_tokens=50) also allow it to ramble.


**2. The model struggles with questions that require *specific factual recall***

Most of the “no_match” cases were for questions that needed a very *exact* medical fact (e.g., a specific structure, enzyme, or drug).

 Why this happens  
- The fine-tuning dataset is not large enough for strong factual memorization.  
- The base Llama 3.2 model isn’t a medically specialized model.  
- If a term never appeared during fine-tuning, it simply improvises.

This explains the **20% overall accuracy** — the model is learning the “style” of medical QA, but not the deep knowledge.



**3. The model is sensitive to prompt formatting and may be mis-parsing the question**

When the model fails, some answers seem unrelated or incomplete.  
This suggests that the model sometimes:

- Misunderstands the start/end of the question  
- Generates before the answer section  
- Doesn’t properly recognize the formatting tokens  

 Why this happens  
If the fine-tuning prompt format slightly differs from the inference prompt, the model becomes confused.  
LLMs are very strict about instruction structure.


 **4. The model tends to hallucinate when unsure**

A consistent failure pattern:  
When the model doesn’t know the answer, it **confidently invents medical facts**.

Examples include:

- listing irrelevant viruses  
- adding random drug combinations  
- giving definitions unrelated to the question  

 Why this happens  
- Not enough medical training data  
- LLMs naturally fill uncertainty with “plausible”-sounding text  
- No restriction mechanism (like constrained decoding)



 **5. Partial matches occur only when key terms appear in the prediction**

The two partial matches suggest one thing:

The model *recognizes the topic* of the question,  
but cannot pinpoint the exact ground truth answer.

This shows that the fine-tuning did help the model learn *context*,  
but not the exact mapping → answer.



 **Summary of Failure Patterns**

Here’s a short summary of what I learned:

- **Over-generation**: listing many diseases instead of one answer  
- **Weak factual precision**: struggles with specific, exact medical facts  
- **Prompt sensitivity**: some formatting mismatches reduce accuracy  
- **Hallucination**: invents long medical lists when unsure  
- **Limited dataset coverage**: small training set → poor generalization  

These failure patterns are consistent with a lightly fine-tuned general-purpose model on a highly specialized medical task.



 **What this means going forward**

Understanding these errors helps me decide what to fix:

- Provide more focused and larger training data  
- Improve prompt consistency  
- Use stronger medical models or domain-specific LoRA adapters  
- Reduce max_new_tokens to control over-generation  

Overall, the model **shows early signs of learning**, but it doesn’t have enough medical depth yet — which fully explains the ~20% accuracy.

**Question 3: Data Quality vs. Quantity**
What do you think it's better between training on 2000 examples (same quality) or 500 curated high-quality examples ?


**1. High-quality data reduces noise and makes the model “learn the right thing”**

If the dataset contains inconsistent answers, unclear phrasing, or slightly incorrect medical information, the model basically learns *noise*.  
With only 500 examples but all perfectly structured, consistent, and medically accurate, the model gets a much clearer signal.

So even if it’s smaller, the quality prevents the model from:

- learning contradictions  
- generating hallucinations  
- copying flawed logic  

This is super important in medical tasks where precision matters.



 **2. LLMs respond surprisingly well to small but clean datasets**

Modern LLMs (even small ones like Llama 3.2) don’t need extremely large fine-tuning datasets to learn a pattern.  
What they need is **clean structure**:

- consistent prompt format  
- consistent answer style  
- no noise  
- no ambiguity  

A dataset of 500 “perfect” examples can outperform 2000 examples that vary in structure or clarity.


 **3. Low-quality quantity can *hurt* performance**

More data is not always better.

If the extra 1500 examples include:

- unclear questions  
- sloppy formatting  
- inconsistent medical terminology  

the model gets confused and the fine-tuning process becomes harder.

You end up with:

- lower accuracy  
- unstable predictions  
- more hallucinations  

Basically, bad data scales the wrong direction.



**4. When quantity *would* matter**

Quantity starts to matter more when the data is both:

- high quality  
- diverse  

So if I had 2000 **curated, consistent, accurate** examples, then yes, quantity would win.

But between:

- **500 excellent examples**, and  
- **2000 examples of average quality**

→ I would absolutely pick the **500 curated** ones.



** Final Answer (my conclusion)**

For this project, I believe **500 curated high-quality examples** are more valuable because the model benefits more from clean, consistent, and reliable data, especially in a domain like medicine where precision is crucial.  

More mediocre examples would likely introduce noise and reduce the model’s ability to produce correct, concise answers — which matches what we saw in our evaluation results.



### Part B : Resource-Constrained Inference

**Question 4: Optimizing for limited resources**
How can you design a strategie to reduce inference time/memory for deployment in constrained environments ?

 **1. Quantize the model**
Convert weights from 16-bit or 32-bit → 8-bit or 4-bit.

**Why it helps:**  
- Huge memory savings  
- Faster matrix multiplications  

**Trade-off:**  
Slight loss of accuracy.


**2. Use a smaller model or distill the current one**
Distill the model into a compact version that keeps the important knowledge.

**Why it helps:**  
- Much faster inference  
- Lower RAM usage  

**Trade-off:**  
Reduced reasoning depth.

**3. Limit max_new_tokens and simplify prompts**
Shorter generation length = faster response.

**Why it helps:**  
- Cuts decode time significantly  

**Trade-off:**  
Answers must be concise.

 **4. Use LoRA adapters instead of full fine-tuned models**
Load only lightweight LoRA weights on top of a base model.

**Why it helps:**  
- Very small memory footprint  
- No need to reload a giant model checkpoint  

**Trade-off:**  
Depends on the base model being available.

**5. Batch requests or cache frequent outputs**
Cache answers to repeated questions or batch multiple queries.

**Why it helps:**  
- Reduces repeated computation  

**Trade-off:**  
Not always applicable to real-time tasks.


**In short:**  
Quantization + smaller models + shorter outputs = the fastest wins for constrained environments.

**Question 5: Speed vs. Accuracy Trade-offs**
Analyze how changing generation parameters affects speed, quality, and consistency 🥸

 **1. max_new_tokens**
- **Lower value → faster, cheaper, but shorter answers**
- **Higher value → slower, more detailed, but risk of rambling**

**Trade-off:**  
Reducing max_new_tokens is the easiest way to speed up inference, but you might cut off medically relevant details.

 **2. temperature**
- **Low (0.0–0.3):** deterministic, stable, but sometimes too rigid  
- **High (0.7+):** more creative, but inconsistent

**Trade-off:**  
Low temperature improves medical accuracy (less hallucination), but may reduce variety.

**3. top_p (nucleus sampling)**
- **Low top_p (0.1–0.5):** safer, more precise, less diverse  
- **High top_p (0.9+):** more fluent but riskier

**Trade-off:**  
Lower top_p improves factual correctness but can sound repetitive.

 **4. Repetition penalty**
- Helps avoid loops or repeated symptoms  
- Slightly increases inference time

**Trade-off:**  
Better readability vs. a small slowdown.

**5. Batch size and GPU settings**
- Larger batch → better throughput but higher memory use  
- Smaller batch → slower but fits on low resources

 ** Summary (student-style)**
If I want **speed**, I lower max_new_tokens + reduce sampling freedom (low temperature + low top_p).  
If I want **accuracy**, I keep temperature low but allow enough max tokens for the model to finish the reasoning.  
If I want **creativity**, I increase temperature/top_p, but this usually hurts correctness for medical tasks.

In short:  
**Fast = short + deterministic**  
**Accurate = longer + controlled sampling**  
**Creative = slower + higher randomness**


### Part C : Evaluation Methodology

**Question 7: Improving Evaluation Metrics**
Analyze limitations of current exact/partial match evaluation and propose improvements. Do you think you have false negatives or false positives ? What can we do about it ?

**1. Limitations of Exact Match**
Exact match is very strict:  
- If the model’s answer contains the correct concept but uses different wording, it is counted as **wrong**.  
- Example: model outputs *“MI (heart attack)”* but ground truth is *“myocardial infarction”* → counted incorrect even though it’s correct.

 This causes **false negatives** (model correct, metric says wrong).



**2. Limitations of Partial Match**
The partial-match rule (≥70% keyword overlap) is simple but also flawed:

 False positives  
If the model outputs a long list of random diseases, it might accidentally include enough keywords to pass the partial-match threshold.

 False negatives  
If the ground truth is short (e.g., “gastric ulcer”), and the model says something very close but missing one key term (e.g., “ulcer”), it gets counted as wrong even though it's semantically close.

 Why this happens  
Keyword overlap ≠ medical correctness.  
Meaning and intent matter more than word matching.

 **3. What improvements can we make?**

 **a. Use semantic similarity (embedding-based evaluation)**
Compute cosine similarity between:
- embedding(model_prediction)  
- embedding(ground_truth)

If similarity > threshold → count as correct.

**Benefits:**  
- Handles synonyms  
- Handles paraphrases  
- Less sensitive to formatting

 **b. Use medical NER (Named Entity Recognition)**
Extract medical entities (diseases, drugs, organs) from both answers.

Match based on entities, not plain text.

**Benefits:**  
- Much more aligned with medical reasoning  
- Avoids evaluating irrelevant filler text

 **c. Use strict normalization**
Before evaluating:
- lowercase  
- remove punctuation  
- remove stopwords  
- convert vocabulary (e.g., “MI” → “myocardial infarction”)

**Benefits:**  
Makes the evaluation fairer and reduces false negatives.
 **d. Human-in-the-loop evaluation for edge cases**
For deployment-level evaluation, ambiguous predictions should be reviewed manually, especially in medicine.

**4. Do we have false negatives or false positives?**

  **False negatives? Yes.**  
The model sometimes outputs the right idea but with different phrasing → counted incorrect by our metric.

  **False positives? Possibly.**  
If the model guesses a long list of conditions and accidentally includes the correct one, partial match may give it credit it doesn’t deserve.

** Summary**
Our exact/partial match metric is simple but shallow.  
We should improve evaluation by:

- adding semantic similarity  
- focusing on medical entities  
- normalizing text  
- combining multiple signals  

This would make accuracy numbers more meaningful and reduce misleading classifications.


**Question 8: Test Set Size and Confidence**
Test other test size and observe the result. What can you say about the results ? How can you improve it ?

 **1. Small test sets give unstable accuracy**
When I tested with only 10–20 examples, the accuracy fluctuated a lot (sometimes 30%, sometimes 20%).  
This makes sense: one or two “easy” or “hard” questions can completely shift the score.

 Conclusion  
**Small test sets produce unreliable estimates.**  
The model might look “good” or “bad” just by luck.



**2. Larger test sets give more stable results**
When I increased the number to 50+ examples, accuracy stabilized and moved less from one run to another.

Conclusion  
**Bigger test sets reduce randomness and give a clearer picture of actual performance.**



 **3. The model remains low-accuracy overall**
Even with bigger batches, the accuracy stayed low (still around 20–30%).  
This means the issue is not only test size — the model itself needs improvement.



 **4. How to improve confidence and reliability**

 **a. Use stratified sampling**
Right now the test examples are sampled randomly.  
If the dataset contains mixed question types, we might accidentally sample only “hard” or “easy” questions.

Stratifying by:
- medical topic  
- question difficulty  
- type of answer  
would produce a more representative evaluation.
 **b. Increase the test size**
Using 100–300 examples would give:
- less variance  
- more trustworthy accuracy  
- better understanding of failure patterns

 **c. Run multiple evaluation rounds and average the results**
Instead of one random sample of 20, run:
- 5 runs of 20  
- or 3 runs of 50  

Then average the accuracy.

This reduces the impact of randomness.

 **d. Improve the evaluation metric**
Since the model sometimes produces partially correct answers, improving the metric (semantic similarity, entity matching, etc.) would reduce false negatives and give a more realistic accuracy.

 **Final takeaway**
Small test sets give noisy, unreliable accuracy numbers.  
Using larger, stratified, and repeated test runs gives a much clearer picture of model performance — and shows the model still needs more training, better prompts, or better fine-tuning data to reach consistent results.


### Part D : Real-World deployment scenario

**Question 9: Production Considerations**
What can you do to address safety, reliability, updates, and edge cases for deploying in a medical assistance application ?

 **1. Safety Mechanisms**
 **a. Add strict refusal rules**
If a question looks like:
- diagnosis  
- medication prescription  
- emergency triage  
the model should refuse and redirect the user to a real doctor.
 **b. Use guardrails / filters**
Before the prompt reaches the model:
- detect harmful intent  
- detect ambiguous or dangerous medical requests  
- block or rephrase risky inputs

This prevents the model from giving unsafe advice.

 **2. Reliability and Monitoring**
 **a. Log all predictions**
Store inputs + outputs so we can detect:
- hallucinations  
- systematic errors  
- risky behaviors  

 **b. Confidence estimation**
Use uncertainty scores or semantic similarity checks to detect when the model is *guessing*.

If confidence is low → the system should warn the user.

 **3. Updating the Model Safely**
 **a. Use versioning**
Keep “Model v1”, “Model v2”, etc.  
Never update a model directly in production without:
- A/B testing  
- regression tests  
- medical validation

 **b. Continuous fine-tuning**
Update the model periodically using:
- curated feedback  
- newly validated medical data  
- new treatment guidelines

But every update must be reviewed by medical professionals.

**4. Handling Edge Cases**
Medical questions can be:
- incomplete  
- contradictory  
- outside the model’s knowledge  
- involving rare diseases  

To handle these:

 Add fallback responses  
If the model doesn’t understand or is unsure → answer with caution or escalate.

 Use intent detection  
If the question is not safe for an LLM (e.g., “My chest hurts right now”), the system must immediately advise contacting emergency services.

 Enforce answer templates  
Structured outputs reduce hallucination, e.g.:

 **5. Human-in-the-loop**
In real medical applications, an LLM *should never operate alone*.  
A clinician must validate the model’s final output for:
- complex cases  
- diagnostics  
- treatment suggestions  
- high-risk decisions  

This drastically reduces risk.

**Final Summary**
To deploy a medical assistant safely, I would combine:
- input filtering  
- output monitoring  
- strict refusal rules  
- model versioning  
- fallback systems  
- human oversight  

The goal is not only accuracy but **predictable, safe, and medically responsible behavior**.
